# Framework Orchestrator

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook orchestrates the end-to-end data ingestion and processing framework. Databricks-ready and production best practices applied.

---

**Instructions:**
1. Configure required parameters using widgets below.
2. Run all cells to execute the full framework orchestration.

---

**Notebook Metadata:**
- **Tested on:** Databricks Runtime 12.2 LTS
- **Contact:** 


In [ ]:
# Databricks widgets for parameterization
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("framework_orchestrator")

dbutils.widgets.text("run_id", "", "Run ID")
run_id = dbutils.widgets.get("run_id")

logger.info(f"Starting orchestration with run_id: {run_id}")

# Example: Call main orchestration logic (replace with actual calls)
try:
    # orchestrate_framework(run_id)
    print(f"Framework orchestration started for run_id: {run_id}")
    # Add more orchestration steps as needed
except Exception as e:
    logger.error(f"Error in orchestration: {e}")
    print(f"Error: {e}")


## Utility Functions

The following cells define utility and helper functions used throughout the orchestrator.

In [ ]:
# --- Utility and Orchestrator Functions ---

def _require_source_fields(source: dict) -> None:
    required = [
        "source_system",
        "source_entity",
        "source_type",
        "landing_table",
        "conformance_table",
        "silver_table",
        "publish_mode",
    ]
    missing = [k for k in required if not source.get(k)]
    if missing:
        raise ValueError(f"Invalid source config for {source}: missing {missing}")


def _connection_profiles(global_config: dict) -> tuple[dict, dict]:
    connections = global_config.get("connections", {})
    return connections.get("jdbc_profiles", {}), connections.get("api_profiles", {})


def _ingest_source(spark, source: dict, global_config: dict):
    source_type = source["source_type"].upper()
    source_options = parse_json_cell(source.get("source_options_json"))
    load_type = str(source.get("load_type", global_config.get("execution", {}).get("default_load_type", "incremental"))).lower()
    jdbc_profiles, api_profiles = _connection_profiles(global_config)

    if source_type == "JDBC":
        profile_name = source.get("jdbc_profile", "")
        jdbc_profile = jdbc_profiles.get(profile_name, {})
        if load_type == "incremental" and source.get("watermark_column") and source_options.get("incremental_start_value"):
            watermark_col = source["watermark_column"]
            start_value = str(source_options["incremental_start_value"]).replace("'", "''")
            table = source.get("jdbc_table", "")
            source = {
                **source,
                "jdbc_table": f"(SELECT * FROM {table} WHERE {watermark_col} > '{start_value}') t",
            }
        return ingest_jdbc_batch(spark=spark, jdbc_profile=jdbc_profile, source=source, extra_options=source_options)

    if source_type == "API":
        profile_name = source.get("api_profile", "")
        api_profile = api_profiles.get(profile_name, {})
        query_params = parse_json_cell(source.get("api_query_params_json"))
        if load_type == "incremental" and source.get("watermark_column") and source_options.get("incremental_start_value"):
            query_params[source["watermark_column"]] = source_options["incremental_start_value"]
        records = ingest_api(source=source, api_profile=api_profile, params=query_params, source_options=source_options)
        return spark.createDataFrame(records)

    if source_type == "FILE":
        file_mode = str(source_options.get("file_ingest_mode", "batch")).lower()
        if file_mode == "autoloader":
            return ingest_file_stream(spark=spark, source=source, global_config=global_config, source_options=source_options)
        return ingest_file_batch(spark=spark, source=source, global_config=global_config, source_options=source_options)

    raise ValueError(f"Unsupported source_type: {source_type}")


---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure orchestration started. Add more tests as needed for your workflow.


In [ ]:
# Simple validation
assert run_id != "", "Run ID widget must be set."
print("Validation passed: Orchestration started.")


---

## Resource Cleanup

Clean up any temporary resources or variables if needed to avoid memory issues in production workflows.